# Current version : 11.A (2025-11-13)

# Libraries and directory (always run)

In [ ]:
### import necessary libraries
# import anndata as ad
from datetime import datetime
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import os
import pandas as pd
import random
import seaborn as sns
import scanpy as sc
import warnings

warnings.filterwarnings("ignore") 
sc.logging.print_header()
sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.verbosity = 1 # errors (0), warnings (1), info (2), hints (3)
plt.rcParams["font.family"] = "Arial"
sns.set_style("white")

pd.options.display.max_rows = 9999

start_time = datetime.now()

def print_with_elapsed_time(message):
    elapsed_time = datetime.now() - start_time
    elapsed_seconds = elapsed_time.total_seconds()
    print(f"[{elapsed_seconds:.2f} seconds] {message}")

In [2]:
print(f"pandas version: {pd.__version__}")
print(f"scanpy version: {sc.__version__}")

In [ ]:
### Directory where the data is stored

# dir = "/mnt/d/Xenium" #Ubuntu
# dir = 'D:/Xenium'
# dir = "/media/volume/data/spatial/hugo/data" #Ubuntu
# dir = "/media/volume/data/spatial/hugo/data/k5" #Ubuntu
# dir = '/media/volume/volume_spatial/hugo/data/test'
dir = '/media/volume/volume_spatial/hugo/data'

# dir_notebook = 'D:/Jupyter_notebook/Xenium_jupyter_notebook'
# dir_notebook = '/mnt/d/Jupyter_notebook/Xenium_jupyter_notebook'
# dir_notebook = '/media/volume/data/spatial/hugo/notebook'
dir_notebook = '/media/volume/volume_spatial/hugo/notebook'


In [ ]:
from module.misc import sample_name_import

name_dir = "circa-SD"
# name_dir = "all-samples"
# name_dir = 'all-samples-C123'

samples, samples_ids = sample_name_import(name_dir)

print(len(samples))
print(samples)

# Data pre-processing

## Import data from Xenium output

In [ ]:
from module.xenium_preprocessing import import_xenium

adata = import_xenium(dir, dir_notebook, samples, samples_ids, name_dir)

In [ ]:
list_noise = pd.read_csv('')

In [ ]:
import pandas as pd
import scanpy as sc
import os
from module.misc import list_annotations

def import_xenium(dir, dir_notebook, samples, samples_ids, name_dir):
    adatas = []
    for sample, sample_id in zip(samples, samples_ids):
        adata = sc.read_10x_h5(f"{dir}/{sample}/cell_feature_matrix.h5")
        df = pd.read_csv(f"{dir}/{sample}/cells.csv.gz")
        df.set_index(adata.obs_names, inplace=True)
        adata.obs = df.copy()
        adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].copy().to_numpy()
        adata.layers["counts"] = adata.X.copy()
        # sc.pp.calculate_qc_metrics(adata,  percent_top=(10, 20, 50, 150), inplace=True)
        # sc.pp.filter_cells(adata, max_counts=1000) ## Possible filter to remove cells with too many transcripts
        sc.pp.filter_cells(adata, min_counts=40) ## Filter cells with less than 40 transcripts
        sc.pp.filter_genes(adata, min_cells=5) ## Filter genes expressed in less than 5 cells
        adata.obs_names = [f"{sample_id}_{cell_id}" for cell_id in adata.obs_names]
        adata.obs['cell_id'] = adata.obs_names
        adatas.append(adata)
        # adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_{sample_id}_forMMC.h5ad")
        print(f"Sample {sample} done")

        if not os.path.exists(f"{dir_notebook}/h5ad/{name_dir}/"):
            os.makedirs(f"{dir_notebook}/h5ad/{name_dir}/")
        adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_{sample}_forMMC.h5ad")

    print(f"Read all {len(samples)} samples")

    ### merge all the anndata objects into a single object
    adata = adatas[0].concatenate(adatas[1:], index_unique=None)

    ### Add a sample column to the metadata
    adata.obs['sample'] = adata.obs_names.map(lambda name: name.split('_')[0])
    # samples = adata.obs['sample'].unique()

    return adata

In [ ]:
# If you don't want to use MMC, skip to next section

# Stop here and run MapMyCell with the individual h5ad generated.
# Then put the .csv files obtained in the "Correlation_Mapping" folder, renamed as: {sample}_CorrelationMapping.csv


In [ ]:
from module.xenium_preprocessing import mmc_merge
adata = mmc_merge(adata, dir_notebook, name_dir)

In [ ]:
adata.obs['mmc:subclass_name'].isna().sum()

# Should be 0. If not zero, some cell are not annotated.
# Check the name of the CorrelationMapping files; if there is any missing, etc.


In [ ]:
if not os.path.exists(f"{dir_notebook}/h5ad/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/h5ad/{name_dir}/")
adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC.h5ad.gz", compression='gzip')

In [ ]:
adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC.h5ad.gz")

## Compute quality metrics

In [ ]:
sc.pp.calculate_qc_metrics(adata,  percent_top=(10, 20, 50, 150), inplace=True)

In [ ]:
cprobes = (adata.obs["control_probe_counts"].sum() / adata.obs["total_counts"].sum() * 100)
cwords = (adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100)
print(f"Negative DNA probe count % : {cprobes}")
print(f"Negative decoding count % : {cwords}")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 3))

axs[0].set_title("Total transcripts per cell")
sns.histplot(adata.obs["total_counts"],kde=True,ax=axs[0])

axs[1].set_title("Unique transcripts per cell")
sns.histplot(adata.obs["n_genes_by_counts"],kde=True,ax=axs[1])

axs[2].set_title("Area of segmented cells")
sns.histplot(adata.obs["cell_area"], kde=True, ax=axs[2])

axs[3].set_title("Nucleus ratio")
sns.histplot(adata.obs["nucleus_area"] / adata.obs["cell_area"], kde=True,ax=axs[3])

if not os.path.exists(f"{dir_notebook}/plot/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/plot/{name_dir}/")
plt.savefig(f"{dir_notebook}/plot/{name_dir}/{name_dir}_quality-metrics.svg")

In [ ]:
for run in adata.obs['run'].unique():
    adata_sub = adata[adata.obs['run']==run]

    fig, axs = plt.subplots(1, 4, figsize=(15, 3))

    axs[0].set_title("Total transcripts per cell")
    sns.histplot(adata_sub.obs["total_counts"],kde=True,ax=axs[0])

    axs[1].set_title("Unique transcripts per cell")
    sns.histplot(adata_sub.obs["n_genes_by_counts"],kde=True,ax=axs[1])

    axs[2].set_title("Area of segmented cells")
    sns.histplot(adata_sub.obs["cell_area"], kde=True, ax=axs[2])

    axs[3].set_title("Nucleus ratio")
    sns.histplot(adata_sub.obs["nucleus_area"] / adata_sub.obs["cell_area"], kde=True,ax=axs[3])

    if not os.path.exists(f"{dir_notebook}/plot/{name_dir}/"):
        os.makedirs(f"{dir_notebook}/plot/{name_dir}/")
    plt.savefig(f"{dir_notebook}/plot/{name_dir}/{run}_{name_dir}_quality-metrics.svg")

## Normalize, PCA, UMAP

In [ ]:
### Normalize, log1p, scale, PCA, and UMAP
start_time = datetime.now()
print_with_elapsed_time(f"Start")
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=None, inplace=True)
print_with_elapsed_time(f"Normalize done")
sc.pp.log1p(adata)
print_with_elapsed_time(f"log1p done")

In [ ]:
# if not os.path.exists(f"{dir_notebook}/h5ad/{name_dir}/"):
#    os.makedirs(f"{dir_notebook}/h5ad/{name_dir}/")
# adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz", compression='gzip')

adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz")

In [ ]:
import scanpy.external as sce

start_time = datetime.now()
print_with_elapsed_time(f"Start")
sc.pp.pca(adata, n_comps = 50)
print_with_elapsed_time(f"PCA done")

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
sc.pl.pca(
    adata,
    color=["sample","sample"],
    dimensions=[(0, 1), (2, 3)],
    ncols=2,
    size=1,
)

In [ ]:
if not os.path.exists(f"{dir_notebook}/h5ad/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/h5ad/{name_dir}/")
adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz", compression='gzip')

# adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz")

In [ ]:
adata.obs['run'] = adata.obs_names.map(lambda name: name.split('-')[0]) ### If multiple runs combined in one dataset. Adapt separation sign to the actual name

if adata.obs['run'].nunique() > 1:
    sce.pp.harmony_integrate(adata, key = 'run', basis = f"X_pca",  adjusted_basis = f"X_pca") ### Will overwrite original pca. Change "adjusted_basis" name to save both.
    print_with_elapsed_time(f"Harmony done")

In [ ]:
if not os.path.exists(f"{dir_notebook}/h5ad/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/h5ad/{name_dir}/")
adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz", compression='gzip')

# adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz")

In [ ]:
start_time = datetime.now()
print_with_elapsed_time(f"Start")
sc.pp.neighbors(adata)
print_with_elapsed_time(f"Neighbors done")
sc.tl.umap(adata, min_dist = 1)
print_with_elapsed_time(f"UMAP done")

In [ ]:
# if not os.path.exists(f"{dir_notebook}/h5ad/{name_dir}/"):
#    os.makedirs(f"{dir_notebook}/h5ad/{name_dir}/")
# adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz", compression='gzip')

adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_MMC_norm.h5ad.gz")

In [ ]:
start_time = datetime.now()
print_with_elapsed_time(f"Start")
sc.tl.leiden(adata, resolution = 2) ### Use a higher resolution value to obtain more clusters. They can be adjusted/merged/subclustered later
print_with_elapsed_time('End of clustering')

In [ ]:
# sc.pl.umap(
#     adata,
#     color="leiden",
#     # Setting a smaller point size to get prevent overlap
#     size=0.25,
# )
sc.pl.umap(adata, color="leiden")

In [ ]:
from module.dataviz_analysis import cluster_plot

cluster_plot(adata,
             name_dir=name_dir,
             dir_notebook=dir_notebook,
             cluster_to_use='leiden')

In [ ]:
adata.obs.drop('leiden_colors', axis=1, inplace=True)
if not os.path.exists(f"{dir_notebook}/h5ad/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/h5ad/{name_dir}/")
adata.write(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_clusters.h5ad.gz", compression='gzip')

In [ ]:
adata = sc.read_h5ad(f"{dir_notebook}/h5ad/{name_dir}/{name_dir}_clusters_bis.h5ad.gz")

## Create a normalised datamatrix

In [ ]:
# Create a normalised datamatrix for saving to disk as a csv file - rows are cells, columns are genes
df = pd.DataFrame(data=adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)
df.shape

from module.xenium_preprocessing import add_annotations

df = add_annotations(adata, df)

### Extract normalized expression and clusters for individual cells
if not os.path.exists(f"{dir_notebook}/csv/{name_dir}/"):
   os.makedirs(f"{dir_notebook}/csv/{name_dir}/")
# df.to_csv(f"{dir_notebook}/csv/{name_dir}/{name_dir}_normalized_counts.csv.gz",
#          compression={'method': 'gzip'})
df.to_parquet(f"{dir_notebook}/csv/{name_dir}/{name_dir}_normalized_counts.parquet")
# adata.obs.to_csv(f"{dir_notebook}/csv/{name_dir}/{name_dir}_MMC_norm.csv")

# End of this notebook

Next step : clusters annotation

[v10B_Xenium_annotation_n_subsclustering](./v10B_Xenium_annotation_n_subsclustering.ipynb)